# 04 — VLM Path *(Optional)*

> **This notebook is optional.** It requires a GPU with at least 6 GB VRAM.  
> No API key. No Ollama. The model runs entirely on your machine.

The main tutorial uses the **LLM path**: PDF → markdown → language model → graph.  
This notebook shows the **VLM path**: PDF → page images → vision-language model → graph.

Same schema. Same graph output. Different extraction engine.

---

## Why the VLM path exists

| | LLM path | VLM path |
|---|---|---|
| **Input** | Markdown text (via docling OCR) | Document images (pages as pixels) |
| **Best for** | Text-heavy, standard layouts | Complex layouts, forms, scanned docs |
| **Inference** | Local (Ollama) or remote API | Local only |
| **API key** | Optional (needed for remote) | Not needed |
| **Data leaves machine?** | Only with remote provider | Never |
| **GPU required** | No (Ollama runs on CPU) | Yes |
| **Model** | gemma4-8k / gpt-4o-mini | NuExtract-2.0 |

**When VLM wins:** scanned PDFs with poor OCR quality, dense tables where layout matters, forms where position carries meaning, air-gapped environments where even Ollama isn't available.

**The key selling point:** no API key and no Ollama. If your documents are sensitive and you can't route them through any external service, VLM is the path.

---
## Part A — Check GPU Memory

NuExtract-2.0 comes in two sizes:

| Model | VRAM needed | Notes |
|---|---|---|
| `numind/NuExtract-2.0-2B` | ~6 GB | Faster, works on most GPUs |
| `numind/NuExtract-2.0-8B` | ~16 GB | More accurate, needs A100/H100 or equivalent |

Run the cell below to see how much GPU memory you have.

In [ ]:
import subprocess, sys

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            total = torch.cuda.get_device_properties(i).total_memory / 1e9
            free  = (torch.cuda.get_device_properties(i).total_memory
                     - torch.cuda.memory_allocated(i)) / 1e9
            print(f"GPU {i}: {torch.cuda.get_device_properties(i).name}")
            print(f"  Total: {total:.1f} GB  |  Free: {free:.1f} GB")
        if total >= 16:
            print("\n✓ Use NuExtract-2.0-8B (MODEL_NAME below)")
        elif total >= 6:
            print("\n✓ Use NuExtract-2.0-2B (MODEL_NAME below)")
        else:
            print("\n✗ Not enough VRAM — use the LLM path (01_quickstart.ipynb)")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("Apple Silicon MPS detected.")
        print("NuExtract-2.0-2B may work but is untested — expect OOM on 18GB models.")
    else:
        print("No GPU detected. VLM path requires a GPU — use 01_quickstart.ipynb instead.")
except ImportError:
    print("torch not installed — cannot check GPU. Install with: pip install torch")

---
## Part B — Choose Model and Schema

Set `MODEL_NAME` to whichever size your GPU can fit.

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME = "numind/NuExtract-2.0-2B"   # change to -8B if you have 16+ GB VRAM
SOURCE_PDF = "../data/sample-declarations-page.pdf"
# ────────────────────────────────────────────────────────────────────────────

from pathlib import Path
assert Path(SOURCE_PDF).exists(), f"PDF not found: {SOURCE_PDF}"
print(f"Model : {MODEL_NAME}")
print(f"Source: {SOURCE_PDF}")

In [ ]:
# Same schema as 01_quickstart — the schema works with both backends unchanged
from pydantic import BaseModel, ConfigDict, Field
from typing import List, Optional

class Insurer(BaseModel):
    model_config = ConfigDict(graph_id_fields=["name"])
    name: str = Field(description="Insurance company name")
    address: Optional[str] = Field(default=None)
    phone: Optional[str] = Field(default=None)

class Insured(BaseModel):
    model_config = ConfigDict(graph_id_fields=["name"])
    name: str = Field(description="Full name(s) of insured")
    address: Optional[str] = Field(default=None)

class Agent(BaseModel):
    model_config = ConfigDict(graph_id_fields=["name"])
    name: str = Field(description="Agent name")
    phone: Optional[str] = Field(default=None)

class Coverage(BaseModel):
    model_config = ConfigDict(graph_id_fields=["coverage_code"])
    coverage_code: str = Field(description="e.g. 'Coverage A', 'Coverage B'")
    coverage_name: str = Field(description="e.g. 'Dwelling', 'Personal Property'")
    limit: Optional[str] = Field(default=None)
    premium: Optional[str] = Field(default=None)

class Deductible(BaseModel):
    model_config = ConfigDict(graph_id_fields=["peril_type"])
    peril_type: str = Field(description="e.g. 'All Other Perils', 'Hurricane'")
    amount: Optional[str] = Field(default=None)

class HomePolicy(BaseModel):
    """Root entity — home insurance declarations page."""
    model_config = ConfigDict(graph_id_fields=["policy_number"])
    policy_number: str = Field(description="e.g. 'FHO295000'")
    policy_form: Optional[str] = Field(default=None)
    effective_date: Optional[str] = Field(default=None)
    expiration_date: Optional[str] = Field(default=None)
    total_premium: Optional[str] = Field(default=None)

    insurer: Optional[Insurer] = Field(default=None)
    insured: Optional[Insured] = Field(default=None)
    agent: Optional[Agent] = Field(default=None)
    coverages: List[Coverage] = Field(default_factory=list,
        description="All coverage lines from the declarations page")
    deductibles: List[Deductible] = Field(default_factory=list)

print("Schema ready. Identical to LLM path — one schema works with both backends.")

---
## Part C — Run the VLM Pipeline

The only change from the LLM config: `backend="vlm"`, `inference="local"`, and `model_override` points to NuExtract.

There is no `provider_override` — VLM has no providers, it loads the model directly from HuggingFace on first run (~4-16 GB download).

> **First run:** the model will be downloaded from HuggingFace. Subsequent runs use the cached copy.  
> **Expected time:** 3–8 min (2B model on GPU). CPU-only will be much slower.

In [ ]:
import warnings, logging
import transformers
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

from docling_graph import run_pipeline, PipelineConfig

config = PipelineConfig(
    source=SOURCE_PDF,
    template=HomePolicy,
    backend="vlm",           # vision-language model backend
    inference="local",       # VLM is always local
    model_override=MODEL_NAME,
    dump_to_disk=True,
)

print(f"Running VLM extraction with {MODEL_NAME} ...")
print("(First run downloads the model — this may take a few minutes)")
ctx = run_pipeline(config)
G = ctx.knowledge_graph
print(f"\nDone. {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

---
## Part D — Explore the Graph

Everything from here is identical to the LLM path — the graph has the same structure.

In [ ]:
from collections import Counter

type_counts = Counter(d.get("__class__") for _, d in G.nodes(data=True))
print("Node types extracted:")
for cls, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<20} × {count}")

In [ ]:
# Coverage table — same query as 01_quickstart
coverages = [
    (d.get("coverage_code"), d.get("coverage_name"), d.get("limit"), d.get("premium"))
    for _, d in G.nodes(data=True)
    if d.get("__class__") == "Coverage"
]

print(f"{'Code':<14} {'Name':<30} {'Limit':<18} {'Premium'}")
print("-" * 72)
for code, name, limit, prem in sorted(coverages):
    print(f"{str(code):<14} {str(name):<30} {str(limit):<18} {prem}")

In [ ]:
# Edge traversal — same as LLM path
root_node = next(
    nid for nid, d in G.nodes(data=True)
    if d.get("__class__") == "HomePolicy"
)

print(f"Root: {root_node}\n")
print("Direct edges from root:")
for _, target, edata in G.out_edges(root_node, data=True):
    relation = edata.get('relation') or '—'
    tdata    = G.nodes[target]
    label    = tdata.get('coverage_code') or tdata.get('name') or tdata.get('peril_type') or target[:20]
    print(f"  --[{relation}]--> {tdata.get('__class__')}: {label}")

---
## Part E — LLM vs VLM: Side-by-Side

| | LLM path | VLM path |
|---|---|---|
| Pipeline | PDF → markdown → LLM | PDF → images → VLM |
| Models | gemma4-8k, gpt-4o-mini | NuExtract-2.0 |
| API key | Needed for remote | Never |
| GPU | Optional | Required |
| Speed (home insurance) | 20s (remote) / 2.7 min (local) | 3–8 min (GPU) |
| Accuracy | High for text | High for complex layouts |
| Chunking | Yes (for multi-page) | No — processes page by page |
| Best fit | Standard PDFs, most documents | Scanned forms, dense tables, sensitive data |

**The graph output is identical.** Same Pydantic schema, same NetworkX DiGraph, same query code. The choice of backend is purely an infrastructure and accuracy decision — it doesn't affect how you use the result.

In [ ]:
# Optional: open interactive graph in browser
import webbrowser
from pathlib import Path

out_dir = Path(ctx.output_dir) if hasattr(ctx, 'output_dir') and ctx.output_dir else Path("../data/test_output")
html_files = list(out_dir.rglob("*.html"))
if html_files:
    out_path = html_files[0]
    print(f"Opening: {out_path}")
    webbrowser.open(out_path.resolve().as_uri())
else:
    print("No HTML output found. Set dump_to_disk=True to generate the interactive graph.")